# 02 – Text Completions & Chat Completions

**Audience:** New Seekr Solutions Architects / Forward-Deployed Engineers / Data Scientists  
**Goal:** By the end of this notebook you can:

- Use the SeekrFlow SDK to call **chat completions**  
- Choose between text completions and chat completions for the right use case  
- Control basic generation parameters (temperature, max tokens, stop sequences)  
- Wrap chat completions into small helper functions you can reuse in demos and prototypes  


**AE Lens: why this matters**

- Demonstrates how tone and role can be controlled in responses.
- Useful for customer-facing demos (executive briefings, support, internal search).
- Streaming responses improve UX and perceived latency in live apps.


## 1. Setup & Client

We’ll start by creating a `SeekrFlow` client, the same way we did in the foundations notebook.


In [1]:
# Load environment variables from .env if present
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass


In [2]:
import os

!pip install seekrai

import os
from google.colab import userdata
os.environ["SEEKR_API_KEY"] = userdata.get("SEEKR_API_KEY")

# 🔐 In real usage, set these OUTSIDE the notebook (shell, Docker env, etc.).
# Here we show it explicitly for training purposes only.

if "SEEKR_API_KEY" not in os.environ:
    os.environ["SEEKR_API_KEY"] = "YOUR_API_KEY_HERE"   # TODO: replace
# Optional: only set if you are overriding the default API URL
# os.environ["SEEKR_BASE_URL"] = "https://flow.seekr.com/v1"    # e.g. "https://flow.seekr.com/v1"


In [3]:
import os
from seekrai import SeekrFlow

# Create a client using environment variables set earlier.
client = SeekrFlow(
    api_key=os.environ.get("SEEKR_API_KEY"),
    base_url=os.environ.get("SEEKR_BASE_URL"),  # falls back to default BASE_URL if None
    supplied_headers={"X-Client-Source": "completions-chat-training-notebook"},
)

client

## 2. Text Completions – Basic Usage

Text completions are prompt-in / text-out calls. They’re useful for single-turn generation
and legacy workflows, while chat completions are better for multi-turn, role-based interactions.

Not all models support text completions, so use a model that advertises a **language** or
**completion** capability in your model catalog.


We start with text completions to show the simplest prompt-in/text-out API. If your environment does not support text completions, skip this cell and use chat completions instead.


In [4]:
# Replace this with a supported text-completions model in your deployment.
COMPLETION_MODEL = "meta-llama/Llama-3.1-8B"  # example placeholder

prompt = (
    "SeekrFlow helps teams build trustworthy AI systems. In two sentences, explain"
    " what that means for a solutions architect."
)

try:
    completion = client.completions.create(
        model=COMPLETION_MODEL,
        prompt=prompt,
        max_tokens=120,
        temperature=0.3,
    )

    choice = completion.choices[0]
    text = getattr(choice, "text", None)
    if text is None and isinstance(choice, dict):
        text = choice.get("text", "")

    print(text)
except NotImplementedError:
    print("Text completions are not implemented in this SDK build. Use chat completions instead.")


Text completions are not implemented in this SDK build. Use chat completions instead.


## 3. Chat Completions – Basic Usage

The most common way you’ll interact with models in SeekrFlow is via **chat completions**:

```python
client.chat.completions.create(...)
```

This API takes a list of messages:

- Each message is a dict with keys like `role` and `content`
- Typical roles are `"system"`, `"user"`, and `"assistant"`

Let’s start with a simple one-turn interaction.


This is the canonical chat request. We pass role-based messages to a chat model and receive a structured response.


In [5]:
# Replace this with a supported chat model in your deployment.
CHAT_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"  # example placeholder

messages = [
    {"role": "system", "content": "You are a helpful assistant for SeekrFlow users."},
    {"role": "user", "content": "In one paragraph, explain what SeekrFlow is."},
]

response = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=messages,
    max_tokens=300,
    temperature=0.7,
)

response

ChatCompletionResponse(id='chatcmpl-babff81e-bd00-4d96-ab9e-95b9ec1d1795', object=<ObjectType.ChatCompletion: 'chat.completion'>, created=1776448457, model='meta-llama/Meta-Llama-3-8B-Instruct', choices=[ChatCompletionChoicesData(index=0, finish_reason=<FinishReason.StopSequence: 'stop'>, message=ChatCompletionMessage(role=<MessageRole.ASSISTANT: 'assistant'>, content="SeekrFlow is a cutting-edge AI-powered job search platform designed to revolutionize the way job seekers find their dream careers. By leveraging advanced natural language processing and machine learning algorithms, SeekrFlow uses a unique matching system to connect job seekers with relevant job openings that align with their skills, experience, and career goals. This innovative approach enables users to receive personalized job recommendations, tailored to their individual needs, and provides a streamlined job search experience that saves time and increases the chances of landing a job that's a perfect fit.", refusal=Non

The exact structure of `response` is defined by `seekrai.types.ChatCompletionResponse`.  
In many cases, you’ll want just the **assistant’s text** from the first choice.

In [6]:
# Inspect the first choice's content.
# Depending on the exact type definitions, this may be a dict-like object or have attributes.
first_choice = response.choices[0]

# Try common patterns – adjust as needed based on your SDK version.
try:
    content = first_choice.message["content"]
except (TypeError, KeyError):
    # Fallback if `message` is a pydantic model or has attributes
    content = getattr(getattr(first_choice, "message", first_choice), "content", first_choice)

print(content)

SeekrFlow is a cutting-edge AI-powered job search platform designed to revolutionize the way job seekers find their dream careers. By leveraging advanced natural language processing and machine learning algorithms, SeekrFlow uses a unique matching system to connect job seekers with relevant job openings that align with their skills, experience, and career goals. This innovative approach enables users to receive personalized job recommendations, tailored to their individual needs, and provides a streamlined job search experience that saves time and increases the chances of landing a job that's a perfect fit.


## 4. Controlling Generation: Temperature, Max Tokens, and Stop Sequences

You can influence how the model responds with parameters like:

- `temperature`: how random/creative the output is (0.0 = deterministic, 1.0 = more varied)  
- `max_tokens`: maximum number of tokens to generate  
- `stop`: list of strings where generation should stop  

Let’s see how changing temperature affects style.


In [7]:
def ask_question(question: str, temperature: float = 0.7) -> str:
    """Helper around client.chat.completions.create for quick experiments."""
    messages = [
        {"role": "system", "content": "You are a concise and friendly technical assistant."},
        {"role": "user", "content": question},
    ]
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=messages,
        max_tokens=300,
        temperature=temperature,
    )
    choice = response.choices[0]
    try:
        content = choice.message["content"]
    except (TypeError, KeyError):
        content = getattr(getattr(choice, "message", choice), "content", choice)
    return content

question = "Describe what a Retrieval-Augmented Generation (RAG) system is, in simple terms."

print("Temperature = 0.0\n")
print(ask_question(question, temperature=0.0))

print("\n" + "="*80 + "\n")

print("Temperature = 0.9\n")
print(ask_question(question, temperature=0.9))

Temperature = 0.0

A Retrieval-Augmented Generation (RAG) system is a type of AI model that combines the strengths of two approaches: retrieval and generation.

**Retrieval**: This part of the system searches through a vast database or knowledge base to find relevant information related to a given input or prompt. Think of it like a super-smart librarian who quickly finds the most relevant books or articles for you.

**Generation**: This part of the system uses the retrieved information to generate new, original text that's tailored to the input or prompt. It's like a creative writer who uses the found information as inspiration to craft a unique story or article.

In a RAG system, the retrieved information is used to augment the generation process, making the output more accurate, informative, and engaging. It's like having a team of experts working together to create high-quality content!

RAG systems are particularly useful for tasks like language translation, text summarization, an

You can also use `stop` sequences to control where the model stops. This is useful when:

- You want to limit the output to a specific section
- You are generating code or data with clear delimiters


In [ ]:
messages = [
    {"role": "system", "content": "Answer in bullet points, each starting with '- '."},
    {"role": "user", "content": "List three benefits of using SeekrFlow."},
]

response = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=messages,
    max_tokens=100,
    temperature=0.7,
    stop=["\n\n"],  # stop at a blank line (example)
)

choice = response.choices[0]
try:
    content = choice.message["content"]
except (TypeError, KeyError):
    content = getattr(getattr(choice, "message", choice), "content", choice)
print(content)

Here are three benefits of using SeekrFlow:


## 5. Streaming Chat Completions

For interactive apps and demos, it’s often nicer to **stream** the response token-by-token (or chunk-by-chunk) instead of waiting for the full completion.

Set `stream=True` to receive an iterator of `ChatCompletionChunk` objects.


Streaming returns incremental chunks. We concatenate the deltas to rebuild the full response for inspection.


In [ ]:
stream_messages = [
    {"role": "system", "content": "You are a streaming demo bot. Respond clearly and succinctly."},
    {"role": "user", "content": "Explain what a SeekrFlow agent is, at a high level."},
]

stream = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=stream_messages,
    max_tokens=300,
    temperature=0.7,
    stream=True,
)

full_text = ""

print("Streaming response:\n")
for chunk in stream:
    choices = getattr(chunk, "choices", None)
    if not choices:
        continue
    # The exact chunk structure may vary; we try a few common patterns.
    try:
        delta = choices[0].delta.get("content") or ""
    except AttributeError:
        # Fallback if delta is an object
        delta_obj = getattr(choices[0], "delta", "")
        delta = getattr(delta_obj, "content", "") if delta_obj else ""

    full_text += delta
    print(delta, end="", flush=True)

print("\n\n---\nFull response captured:")
print(full_text)


Streaming response:

A SeekrFlow agent is a software agent that uses artificial intelligence and machine learning to automate and optimize business processes, workflows, and data flows within an organization. It's designed to streamline and integrate various systems, applications, and data sources to improve efficiency, reduce errors, and enhance decision-making.

---
Full response captured:
A SeekrFlow agent is a software agent that uses artificial intelligence and machine learning to automate and optimize business processes, workflows, and data flows within an organization. It's designed to streamline and integrate various systems, applications, and data sources to improve efficiency, reduce errors, and enhance decision-making.


## 6. System Prompts and Style Control

The **system message** sets the overall behavior and tone of the assistant. This is one of the primary levers you’ll use when building demos and PoCs.

Try changing the system prompt to:

- Make the assistant more formal or casual  
- Target a specific audience (e.g., data engineers vs. product managers)  
- Enforce response formatting rules (e.g., always respond in Markdown)


In [ ]:
def ask_with_style(question: str, style: str) -> str:
    system_prompt = f"""You are an assistant helping users learn SeekrFlow.
You should answer in the following style: {style}
"""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=messages,
        max_tokens=300,
        temperature=0.7,
    )
    choice = response.choices[0]
    try:
        content = choice.message["content"]
    except (TypeError, KeyError):
        content = getattr(getattr(choice, "message", choice), "content", choice)
    return content

question = "How would you explain SeekrFlow to a new Solutions Architect?"

print("Style: Very formal, documentation-like\n")
print(ask_with_style(question, "very formal, like internal documentation"))

print("\n" + "="*80 + "\n")

print("Style: Friendly, conversational, using analogies\n")
print(ask_with_style(question, "friendly, conversational, using simple analogies"))

Style: Very formal, documentation-like

**SeekrFlow Overview for Solutions Architects**

**Purpose:**
This document provides an introduction to SeekrFlow, a cutting-edge platform designed for modern software development and delivery. As a Solutions Architect, this knowledge will enable you to effectively design, implement, and optimize SeekrFlow-based solutions for your clients.

**What is SeekrFlow?**
SeekrFlow is a cloud-native, containerized, and microservices-based platform that enables the development of scalable, secure, and highly available software systems. Its primary focus is on streamlining the development and delivery of software applications, ensuring faster time-to-market, and reducing costs.

**Key Components:**

1. **Core Services:** SeekrFlow provides a suite of pre-built, reusable services that can be easily integrated into applications. These services include authentication, authorization, caching, and more.
2. **Containerization:** SeekrFlow leverages containerizati

## 7. Wrapping Chat Completions into a Helper Function

In real projects and demos, you rarely call `client.chat.completions.create` directly everywhere. Instead, it’s useful to wrap it in a small helper with your preferred defaults.

This also makes it easier to switch models or tweak parameters later.


In [ ]:
def chat_answer(
    question: str,
    system_prompt: str = "You are a helpful assistant for SeekrFlow demos.",
    model: str = CHAT_MODEL,
    max_tokens: int = 300,
    temperature: float = 0.3,
) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature,
    )
    choice = response.choices[0]
    try:
        content = choice.message["content"]
    except (TypeError, KeyError):
        content = getattr(getattr(choice, "message", choice), "content", choice)
    return content


# Quick test
print(chat_answer("Give me three bullet points on why SeekrFlow is useful for enterprises."))

Here are three bullet points on why SeekrFlow is useful for enterprises:

• **Streamlined Information Discovery**: SeekrFlow's AI-powered search engine helps employees quickly find the information they need, reducing the time spent searching for documents, data, and knowledge. This increases productivity and efficiency, allowing employees to focus on higher-value tasks.

• **Enhanced Collaboration and Knowledge Sharing**: SeekrFlow's intuitive interface and advanced search capabilities enable teams to collaborate more effectively, share knowledge, and reduce information silos. This leads to better decision-making, improved innovation, and increased employee engagement.

• **Compliance and Governance**: SeekrFlow's advanced search and analytics capabilities help enterprises maintain compliance with regulatory requirements and internal policies. By providing a centralized platform for information management, SeekrFlow reduces the risk of data breaches, ensures data integrity, and support

## 8. Exercise (for New Hires)

> Try this yourself in the cell below.

1. Implement a function `answer_for_role(question: str, role: str) -> str` that:
   - Uses a system prompt tailored to the given `role` (e.g., `"executive"`, `"data engineer"`, `"product manager"`)
   - Returns a **short** answer (2-3 sentences) using `client.chat.completions.create`
2. Call it with the same question but different roles and compare the outputs.

Hints:

- Reuse the patterns from `chat_answer` or `ask_with_style`
- Keep `temperature` low (e.g., 0.2–0.4) for more deterministic answers


In [ ]:
# Your implementation here

def answer_for_role(question: str, role: str) -> str:
    """Return a short explanation of SeekrFlow tailored to a specific audience role.

    Examples of roles: "executive", "data engineer", "product manager".
    """
    # TODO: implement using client.chat.completions.create
    raise NotImplementedError()

# Example usage (once implemented):
# print(answer_for_role("What is SeekrFlow?", "executive"))
# print(answer_for_role("What is SeekrFlow?", "data engineer"))

## 9. Summary

You’ve now seen how to:

- Call `client.chat.completions.create` with messages and a model  
- Choose text vs chat completions based on interaction style  
- Control output style with `temperature`, `max_tokens`, and `stop`  
- Stream responses chunk by chunk  
- Wrap chat completions into reusable helper functions  

These patterns are the building blocks for:

- Internal CLI tools
- Simple web/Streamlit demos
- The prompting layer inside more advanced agents and workflows

In the next notebook, we’ll connect this to **embeddings and retrieval** to start moving towards RAG-style applications.


## 10. Exercise Solution (Post Summary)

Example implementation using a tailored system prompt per role.

```python
from typing import Optional


def answer_for_role(question: str, role: str) -> str:
    system = (
        f"You are a {role}. Answer in 2-3 concise sentences. "
        "Be clear and non-technical unless asked."
    )
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": question},
        ],
        temperature=0.3,
        max_tokens=120,
    )
    return response.choices[0].message.content.strip()


# Example usage
# print(answer_for_role("What is SeekrFlow?", "executive"))
# print(answer_for_role("What is SeekrFlow?", "data engineer"))
```
